# Lista 4

## Numpy zaawansowane

(5pkt + 2pkt)

Na liście znajduje się 5 zadania do wykonania w trakcie zajęć. Po rozwiązaniu zadania pokaż kod prowadzącemu i odpowiedz na **pytanie kontrolne** — tylko wtedy przyznajemy punkty. Dodatkowo prześlij zadanie na platformie skos.

Jeśli w poleceniu nie jest wskazane użycie pętli `for` - nie korzystaj z niej.

Zadania domowe są oznaczone 🏠. To zadania o numerach 6, 7. Punkty zostaną przyznane tylko za rozwiązania z 3 zadań. Rozwiąż te, które najbardziej Ci się podobają.

Każde zadanie warte jest 1pkt.

## 1. Jak CNN „przesuwa okno” po obrazie?

**Kontekst:** W konwolucyjnej sieci neuronowej filtr (np. 3×3) „przesuwa się” po obrazie piksel po pikselu. Parametr stride określa, o ile pikseli filtr przesuwa się w pamięci między kolejnymi pozycjami. Zrozumienie, jak numpy.strides działa, pozwala intuicyjnie pojąć, dlaczego operacja konwolucji może być wydajna bez kopiowania danych.

**Polecenie**

1. Utwórz: `A = np.arange(1,13).reshape(3,4)` — traktuj jako obraz `3 × 4`, oraz `AT = A.T`.
2. Wypisz: `A`, `A.strides`, `AT`, `AT.strides`.
3. Oblicz ile bajtów trzeba przeskoczyć, aby dotrzeć:
   * od `A[0,0]` do `A[0,1]`,
   * od `A[0,0]` do `A[1,0]`.
4. Sprawdź, czy `AT` jest widokiem (`AT.base is A`), zmień `A[0,0] = -99` i sprawdź `AT`.
5. Napisz funkcję `memory_offset(A, i, j)`, która obliczy ile bajtów trzeba przejść od `A[0][0]` do miejsca `A[i][j]`
   Użyj jej, żeby policzyć offsety kilku elementów w `A` i `A.T`. Zinterpretuj wyniki.

**Pytanie kontrolne:** Wyjaśnij własnymi słowami: co się stanie jak zmienisz `strides`?

In [2]:
import numpy as np

A = np.arange(1, 13).reshape(3, 4)
print("A:\n", A)
print("A.strides:", A.strides) 

AT = A.T
print("AT:\n", AT)
print("AT.strides:", AT.strides)

# elementsize (liczba bajtów na element)
bs = A.itemsize
print("itemsize (bajtów per element):", bs)

# obliczenia ręczne:
offset_A_00_to_01 = A.strides[1]
offset_A_00_to_10 = A.strides[0]
print("offset A[0,0] -> A[0,1] (bajtów):", offset_A_00_to_01)
print("offset A[0,0] -> A[1,0] (bajtów):", offset_A_00_to_10)

# sprawdzenie widoku
print("A.T.base is A ?", AT.base is A)

# zmiana A i obserwacja AT
A[0,0] = -99
print("A po zmianie:\n", A)
print("A.T po zmianie:\n", AT)

# funkcja memory_offset
def memory_offset(M, i, j):
    return i * M.strides[0] + j * M.strides[1]

print("memory offset A[2,3]:", memory_offset(A, 2, 3))
print("memory offset AT[2,3]:", memory_offset(AT, 2, 3))

A:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
A.strides: (32, 8)
AT:
 [[ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]
 [ 4  8 12]]
AT.strides: (8, 32)
itemsize (bajtów per element): 8
offset A[0,0] -> A[0,1] (bajtów): 8
offset A[0,0] -> A[1,0] (bajtów): 32
A.T.base is A ? False
A po zmianie:
 [[-99   2   3   4]
 [  5   6   7   8]
 [  9  10  11  12]]
A.T po zmianie:
 [[-99   5   9]
 [  2   6  10]
 [  3   7  11]
 [  4   8  12]]
memory offset A[2,3]: 88
memory offset AT[2,3]: 112


## 2. Sortowanie i unikalne wartości

**Kontekst:**
Masz wektor `scores` z wynikami modelu klasyfikującego 100 obserwacji.
Chcesz zobaczyć, które próbki miały najwyższe wyniki i jakie unikalne klasy się pojawiły.

**Polecenie:**

* Wygeneruj losowe `scores` (np. `np.random.rand(100)`).
* Posortuj `scores` rosnąco.
* Zwróć **indeksy** sortowania (czyli w jakiej kolejności należy wziąć próbki).
* Wygeneruj losowy wektor `labels` z wartościami z zakresu `[0, 3]`.
* Wypisz unikalne etykiety i ich liczność.

**Podpowiedź:**
`np.sort`, `np.argsort`, `np.unique(..., return_counts=True)`

In [2]:
# dane
scores = np.random.rand(100)
labels = np.random.randint(0, 4, size=100)

# sortowanie
sorted_scores = np.sort(scores)
sorted_idx = np.argsort(scores)

# unikalne wartości
unique_labels, counts = np.unique(labels, return_counts=True)

print("Posortowane wyniki:", sorted_scores[:5], "...")
print("Indeksy sortowania:", sorted_idx[:5])
print("Unikalne etykiety:", unique_labels)
print("Liczność:", counts)

Posortowane wyniki: [0.00253562 0.02303967 0.02440106 0.04076941 0.06155384] ...
Indeksy sortowania: [65 49 56 89 64]
Unikalne etykiety: [0 1 2 3]
Liczność: [18 20 34 28]


## 3. Maski i filtrowanie danych (`numpy.ma`)

**Kontekst:**
Masz dane `loss_values` z treningu modelu (np. z 200 epok).
Chcesz sprawdzić, w których momentach **strata spadła poniżej 0.1**, i wygodnie pracować tylko z tymi fragmentami — używając **masked arrays** (`numpy.ma`), które pozwalają ukryć niechciane wartości bez ich usuwania z tablicy.

**Polecenie:**

* Ustaw random seed.
* Wygeneruj `loss_values` – malejący wykres strat (np. `np.linspace(1, 0, 200) + np.random.normal(0, 0.05, 200)`).
* Utwórz **masked array**, która ukryje wartości większe lub równe `0.1`.
* Wyświetl tylko wartości, które **nie są zamaskowane**.
* Policz, w ilu epokach model osiągnął niski błąd (`loss < 0.1`).
* Oblicz średnią z „dobrych” epok, ignorując zamaskowane dane.

**Podpowiedź:**

Użyj: `ma.masked_greater_equal`


In [3]:
import numpy as np
import numpy.ma as ma

# dane

# Ustaw random seed 
np.random.seed(42)

# Uzupełnij 
loss_values = np.linspace(1, 0, 200) + np.random.normal(0, 0.05, 200)

# tworzymy masked array – maskujemy wszystkie wartości >= 0.1
loss_masked = ma.masked_greater_equal(loss_values, 0.1)

# wyświetlamy kilka niezamaskowanych wartości
print("Wartości z niskim błędem:", loss_masked.compressed()[:5])

# liczymy, ile epok miało stratę poniżej 0.1
n_low_loss = np.sum(loss_values < 0.1)
print("Liczba epok z niskim błędem:", n_low_loss)

# obliczamy średnią tylko z niezamaskowanych danych
avg_low_loss = np.mean(loss_masked)
print("Średnia strata (dla niskich wartości):", round(avg_low_loss, 4))

Wartości z niskim błędem: [0.099913   0.0922948  0.04759438 0.03188251 0.06420375]
Liczba epok z niskim błędem: 19
Średnia strata (dla niskich wartości): 0.0316


## 4. Stride tricks

**Kontekst:**
Podczas trenowania modeli w *computer vision* w celu optymalizacji złożoności modelu obrazy dzieli się na mniejsze fragmenty (np. do uczenia na patchach).

**Polecenie:**

Masz obraz o wymiarach **3 × 1000** (obraz czarno-biały o wymiarach 3x1000).

* Wygeneruj dane losowe (np. `np.random.rand(3, 1000)`).
* Użyj `numpy.lib.stride_tricks.sliding_window_view`, by stworzyć macierz okien, tak żeby obrazy nachodziły na siebie co 10 próbek.
* Sprawdź wymiary wynikowej tablicy. Ile masz okien?

**Podpowiedź:** Rozwiązanie powinno być postaci `sliding_window_view(image, window_shape=rozmiar_okna, axis=...)[::krok]`

In [21]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

# dane wejściowe
image = np.random.rand(3, 1000)

# okna o długości 100 pikseli, przesuwane co 10
windows = sliding_window_view(image, window_shape=100, axis=1)[::90]
print("Wymiary wynikowej tablicy:", windows.shape)
print("Liczba okien:", windows.shape[1])

Wymiary wynikowej tablicy: (1, 901, 100)
Liczba okien: 901


## 5. Maskowanie warunkowe w macierzach i analiza różnic

**Kontekst:**
Masz dwie macierze wyników modeli `A` i `B` `(1000, 10)`.
Chcesz policzyć różnice i średnie, **ignorując**:

* wszystkie pozycje, w których `A` lub `B` mają NaN,
* wartości ekstremalne (>95 percentyl każdej kolumny),
* wartości ujemne w obu macierzach **jednocześnie**.

**Polecenie:**

1. Wygeneruj `A` i `B` z losowego rozkładu i wprowadź kilka `NaN`.
2. Utwórz maskę, która jednocześnie:

   * maskuje NaN (`masked_invalid`),
   * maskuje wartości > 95 percentyl (`masked_greater`),
   * maskuje pozycje, gdzie `A < 0 & B < 0` (`masked_where` z `logical_and`).
3. Stwórz masked array, która będzie  `diff = |A-B|` z utworzoną powyżej maską.
4. Policz średnią i wariancję różnic macierzy `diff`, ignorując zamaskowane wartości.
5. Zastąp zamaskowane wartości średnią kolumnową.

**Podpowiedź**

Z biblioteki `numpy.ma` mogą się przydać:
`masked_invalid`, `masked_greater`, `masked_where`, `mask_or`, `mean`, `var`, `filled`.

Z biblioteki `numpy` przydadzą się: `np.percentile`, `np.logical_and`.

In [ ]:
A = np.random.randn(1000, 10)
B = np.random.randn(1000, 10)

A[0, 0] = np.nan
A[10, 3] = np.nan
B[5, 1] = np.nan
B[20, 7] = np.nan

invalid_mask = np.logical_or(np.isnan(A), np.isnan(B))
extreme_mask = np.logical_or(A > np.percentile(A, 95, axis=0), B > np.percentile(B, 95, axis=0))
negative_mask = np.logical_and(A < 0, B < 0)

mask = invalid_mask | extreme_mask | negative_mask

diff = ma.masked_array(np.abs(A - B), mask=mask)

print(diff.mean())
print(diff.var())

column_means = diff.mean(axis=0).filled(np.nan)
diff_filled = np.where(diff.mask, column_means, diff.filled(0))

print(column_means)
print(diff_filled.shape)

1.1874949547996758
0.6783129051251413
[1.16692283 1.27055554 1.12235166 1.27722492 1.13973542 1.13078073
 1.13581978 1.19981372 1.20413735 1.22018888]
(1000, 10)


## 6. 🏠 Normalizacja niestandardowa

Utwórz `X` o wymiarach `(100, 10)` – 100 obserwacji, 10 cech o różnych zakresach, z zakresu [-100, 100].
Znormalizuj każdą kolumnę **do przedziału od min do max, ale tylko dla wartości dodatnich**.
Wartości ujemne mają pozostać niezmienione.

**Polecenie:**

* Wygeneruj dane `X` z rozkładu normalnego (`np.random.randn(100, 10)`).
* Utwórz maskę `mask = X > 0`.
* Zastosuj broadcastowanie, aby znormalizować każdą kolumnę:

  ```python
  X_norm = (X - X.min(axis=?)) / (X.max(axis=?) - X.min(axis=?))
  ```
* Nałóż maskę tak, aby wartości `≤ 0` w oryginale nie były zmieniane.

**Podpowiedź:**
Użyj `np.where`.

In [3]:
np.random.seed(42)

scales = np.linspace(1, 50, 10)
X = np.clip(np.random.randn(100, 10) * scales, -100, 100)

mask = X > 0

X_min = X.min(axis=0)
X_max = X.max(axis=0)
X_norm = (X - X_min) / (X_max - X_min)

X_result = np.where(mask, X_norm, X)

print("X_min:", X_min.round(2))
print("X_max:", X_max.round(2))
print("Ujemne niezmienione:", np.array_equal(X_result[~mask], X[~mask]))
print("Dodatnie w [0, 1]:", X_result[mask].min() >= 0, X_result[mask].max() <= 1)
print(X_result[:5].round(2))

X_min: [  -1.92  -14.83  -38.54  -38.11  -59.67  -62.4   -90.8   -94.8  -100.
  -99.38]
X_max: [  2.53  13.37  30.44  42.7   58.62  74.29  63.5   96.03 100.   100.  ]
Ujemne niezmienione: True
Dodatnie w [0, 1]: True True
[[  0.54  -0.89   0.67   0.8   -5.33  -6.61   0.93   0.65 -20.92   0.63]
 [ -0.46  -3.     0.6  -33.16 -39.29 -15.87 -34.1    0.56 -40.46 -70.62]
 [  0.76  -1.46   0.57 -24.7  -12.4    0.48 -38.75   0.57 -26.76 -14.58]
 [ -0.6    0.95  -0.16 -18.33   0.66 -34.45   0.63 -76.64 -59.18   0.55]
 [  0.6    0.57  -1.37  -5.22 -33.68 -20.32 -15.51   0.71   0.58 -88.15]]


## 7. 🏠 Stride tricks – zamiana kwadrantów tablicy

Stwórz dwuwymiarową tablicę. **Zamień miejscami górny prawy i dolny lewy kwadrant**, używając tylko `numpy` i `stride_tricks`, bez kopiowania danych.

**Polecenie:**

* Utwórz tablicę `A = np.arange(16).reshape(4, 4)`.
* Za pomocą `stride_tricks` (np. `as_strided`) stwórz nowy widok tablicy. Twoje rozwiązanie powinno być postaci: `stride_tricks.as_strided(...).reshape(...)`

(obrazek w pliku z obrazkami do wykładu)

![task](./Images/ex_np_as_strided.png)


In [5]:
from numpy.lib import stride_tricks

A = np.arange(16).reshape(4, 4)
print("A:\n", A)

h = A.shape[0] // 2
s0, s1 = A.strides

view = stride_tricks.as_strided(
    A,
    shape=(2, h, 2, h),
    strides=(h * s1, s0, h * s0, s1),
)
print("Widok dzieli pamięć z A:", np.shares_memory(view, A))

B = view.reshape(4, 4)
print("Po zamianie kwadrantów:\n", B)

A:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]]
Widok dzieli pamięć z A: True
Po zamianie kwadrantów:
 [[ 0  1  8  9]
 [ 4  5 12 13]
 [ 2  3 10 11]
 [ 6  7 14 15]]
